# 3.1 — What Is an Agent?

A **chain** follows a fixed sequence of steps every time.
An **agent** decides its own steps based on what it observes.

```
CHAIN:  Input → Step 1 → Step 2 → Step 3 → Output   (fixed path)

AGENT:  Input → Think → Act → Observe → Think → Act → Observe → ... → Output
                 ↑___________________________________|   (dynamic loop)
```

The pattern agents use is called **ReAct** (Reason + Act):
- **Thought** — the model reasons about what to do next
- **Action** — the model calls a tool
- **Observation** — the tool returns a result
- Repeat until the model has enough to answer

In [ ]:
!pip install langchain langchain-ollama --quiet

## 1. Chain vs Agent — Side by Side

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model='llama3.1', temperature=0)

# CHAIN — always does exactly two steps, no matter what
step1 = ChatPromptTemplate.from_template('Translate to French: {text}') | llm | StrOutputParser()
step2 = ChatPromptTemplate.from_template('Make this enthusiastic: {text}') | llm | StrOutputParser()

def run_chain(text):
    french = step1.invoke({'text': text})
    excited = step2.invoke({'text': french})
    return excited

print('CHAIN result:')
print(run_chain('The weather is nice today'))
print()
print('Notice: the chain always translates then makes enthusiastic — it cannot skip or add steps.')

In [ ]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage, AIMessage

# Define tools
@tool
def add(a: int, b: int) -> int:
    """Adds two numbers."""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """Multiplies two numbers."""
    return a * b

@tool
def get_capital(country: str) -> str:
    """Returns the capital city of a country."""
    capitals = {
        'france': 'Paris', 'germany': 'Berlin', 'japan': 'Tokyo',
        'india': 'New Delhi', 'usa': 'Washington D.C.', 'brazil': 'Brasilia'
    }
    return capitals.get(country.lower(), f'Unknown capital for {country}')

tools = [add, multiply, get_capital]
tool_map = {t.name: t for t in tools}
llm_with_tools = llm.bind_tools(tools)

# AGENT — decides which tools to call based on the question
def run_agent(question: str):
    print(f'Question: {question}')
    messages = [
        SystemMessage(content='You are a helpful assistant. Use tools when needed.'),
        HumanMessage(content=question)
    ]
    step = 0
    while True:
        response = llm_with_tools.invoke(messages)
        messages.append(response)
        if not response.tool_calls:
            print(f'Answer: {response.content}')
            return
        for tc in response.tool_calls:
            step += 1
            result = tool_map[tc['name']].invoke(tc['args'])
            print(f'  Step {step}: called {tc["name"]}({tc["args"]}) → {result}')
            messages.append(ToolMessage(content=str(result), tool_call_id=tc['id']))

print('AGENT results:')
print()
run_agent('What is 15 + 27?')
print()
run_agent('What is the capital of Japan?')
print()
run_agent('If I have 8 groups of 7 apples, how many apples total?')

## 2. The ReAct Loop — Visualised

```
User: "What is (12 + 8) × 5?"

  Thought:  I need to add 12 and 8 first, then multiply.
  Action:   add(12, 8)  →  Observation: 20

  Thought:  Now I multiply 20 by 5.
  Action:   multiply(20, 5)  →  Observation: 100

  Thought:  I have the answer.
  Answer:   (12 + 8) × 5 = 100
```

The key insight: **the agent loops until it decides it is done.**
A chain would need this exact sequence hardcoded.

In [ ]:
# Multi-step: requires TWO tool calls in sequence
run_agent('What is (12 + 8) multiplied by 5?')

## 3. When to Use a Chain vs an Agent

| Situation | Use |
|-----------|-----|
| Steps are always the same | **Chain** |
| Number of steps is fixed | **Chain** |
| Need speed and predictability | **Chain** |
| Steps depend on the question | **Agent** |
| May need to retry or backtrack | **Agent** |
| Needs to combine multiple tools | **Agent** |

## Summary

| Concept | Description |
|---------|-------------|
| **Chain** | Fixed pipeline — always same steps |
| **Agent** | Dynamic loop — decides steps at runtime |
| **ReAct** | Thought → Action → Observation cycle |
| **Tool** | Function the agent can call |
| **Tool Map** | Dict mapping tool name → function |
| **ToolMessage** | Passes tool result back to the model |